# 03 - Model Evaluation

Detailed evaluation of the trained model including:
- Per-class mAP analysis
- Confusion matrix
- Failure case analysis
- Performance at different confidence thresholds

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

sns.set_theme(style='whitegrid')

# Paths - adjust as needed
try:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_PATH = '/content/drive/MyDrive/construction-safety-monitor/models/best.pt'
    DATASET_DIR = Path('/content/dataset')
except ImportError:
    MODEL_PATH = '../models/best.pt'
    DATASET_DIR = Path('../data/dataset')

DATA_YAML = str(DATASET_DIR / 'data.yaml')

CLASS_NAMES = ['worker', 'hardhat', 'no_hardhat', 'vest', 'no_vest']

# Load model
model = YOLO(MODEL_PATH)
print(f'Model loaded: {MODEL_PATH}')

## 1. Validation Metrics

In [ ]:
# Run validation
val_results = model.val(data=DATA_YAML, split='test', verbose=True)

print('\n' + '='*50)
print('OVERALL METRICS')
print('='*50)
print(f'mAP@50:      {val_results.box.map50:.4f}')
print(f'mAP@50-95:   {val_results.box.map:.4f}')
print(f'Precision:   {val_results.box.mp:.4f}')
print(f'Recall:      {val_results.box.mr:.4f}')

In [ ]:
# Per-class metrics
print('\n' + '='*50)
print('PER-CLASS METRICS')
print('='*50)
print(f'{"Class":<15} {"Precision":>10} {"Recall":>10} {"mAP@50":>10} {"mAP@50-95":>10}')
print('-' * 55)

per_class_map50 = val_results.box.maps  # per-class mAP@50-95
ap50 = val_results.box.ap50  # per-class AP@50 if available

for i, name in enumerate(CLASS_NAMES):
    if i < len(per_class_map50):
        p = val_results.box.p[i] if hasattr(val_results.box, 'p') else 0
        r = val_results.box.r[i] if hasattr(val_results.box, 'r') else 0
        m50 = ap50[i] if i < len(ap50) else 0
        m = per_class_map50[i]
        print(f'{name:<15} {p:>10.4f} {r:>10.4f} {m50:>10.4f} {m:>10.4f}')

In [ ]:
# Per-class mAP bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(CLASS_NAMES))
width = 0.35

if len(ap50) >= len(CLASS_NAMES):
    bars1 = ax.bar(x - width/2, ap50[:len(CLASS_NAMES)], width, label='mAP@50', color='steelblue')
    bars2 = ax.bar(x + width/2, per_class_map50[:len(CLASS_NAMES)], width, label='mAP@50-95', color='coral')
else:
    bars2 = ax.bar(x, per_class_map50[:len(CLASS_NAMES)], width, label='mAP@50-95', color='coral')

ax.set_ylabel('mAP')
ax.set_title('Per-Class Mean Average Precision')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Confusion Matrix

In [ ]:
# The confusion matrix is saved automatically during validation
from PIL import Image

cm_path = Path(val_results.save_dir) / 'confusion_matrix_normalized.png'
if cm_path.exists():
    fig, ax = plt.subplots(figsize=(10, 8))
    cm_img = Image.open(cm_path)
    ax.imshow(cm_img)
    ax.axis('off')
    ax.set_title('Normalized Confusion Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f'Confusion matrix not found at {cm_path}')
    print('Available files:', list(Path(val_results.save_dir).glob('*')))

## 3. Confidence Threshold Analysis

In [ ]:
# Evaluate at different confidence thresholds
thresholds = [0.25, 0.35, 0.45, 0.50, 0.60, 0.70, 0.80]
threshold_results = []

for conf in thresholds:
    r = model.val(data=DATA_YAML, split='test', conf=conf, verbose=False)
    threshold_results.append({
        'conf': conf,
        'precision': r.box.mp,
        'recall': r.box.mr,
        'map50': r.box.map50,
        'map': r.box.map,
    })
    print(f'conf={conf:.2f}: P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

confs = [r['conf'] for r in threshold_results]
axes[0].plot(confs, [r['precision'] for r in threshold_results], 'b-o', label='Precision')
axes[0].plot(confs, [r['recall'] for r in threshold_results], 'r-o', label='Recall')
axes[0].set_xlabel('Confidence Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision-Recall vs Confidence')
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(confs, [r['map50'] for r in threshold_results], 'g-o', label='mAP@50')
axes[1].plot(confs, [r['map'] for r in threshold_results], 'm-o', label='mAP@50-95')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('mAP')
axes[1].set_title('mAP vs Confidence')
axes[1].legend()
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Failure Case Analysis

In [ ]:
# Find images where model struggles (low confidence or missed detections)
import random

test_dir = DATASET_DIR / 'test' / 'images'
if not test_dir.exists():
    test_dir = DATASET_DIR / 'valid' / 'images'

test_images = list(test_dir.glob('*'))

# Collect predictions with low confidence
low_conf_images = []
high_conf_images = []

for img_path in random.sample(test_images, min(100, len(test_images))):
    results = model(str(img_path), conf=0.3, verbose=False)
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        confs = results[0].boxes.conf.cpu().numpy()
        avg_conf = confs.mean()
        if avg_conf < 0.5:
            low_conf_images.append((img_path, avg_conf, results[0]))
        elif avg_conf > 0.8:
            high_conf_images.append((img_path, avg_conf, results[0]))

# Show hardest cases
low_conf_images.sort(key=lambda x: x[1])
if low_conf_images:
    fig, axes = plt.subplots(1, min(4, len(low_conf_images)), figsize=(20, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, (path, conf, res) in zip(axes, low_conf_images[:4]):
        annotated = res.plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Avg conf: {conf:.2f}', fontsize=10)
        ax.axis('off')
    plt.suptitle('Challenging Cases (Low Confidence)', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Show best cases
high_conf_images.sort(key=lambda x: x[1], reverse=True)
if high_conf_images:
    fig, axes = plt.subplots(1, min(4, len(high_conf_images)), figsize=(20, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, (path, conf, res) in zip(axes, high_conf_images[:4]):
        annotated = res.plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Avg conf: {conf:.2f}', fontsize=10)
        ax.axis('off')
    plt.suptitle('Best Cases (High Confidence)', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Observations

### Where the Model Performs Well
- Document strong performance scenarios after running evaluation

### Where the Model Struggles
- Document failure cases and their likely causes

### Recommendations
- Suggested improvements based on evaluation results